In [1]:
import numpy as np
import os

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import SAC
from stable_baselines3.her import HerReplayBuffer, GoalSelectionStrategy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv

from causal_gym import AntMazePCH

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
class AntMazeSCMGoalWrapper(gym.Env):
    def __init__(self, scm_env):
        super().__init__()
        self.env = scm_env
        self.success_radius = self.env.success_radius

        self.obs_keys = list(self.env.observation_space.spaces.keys())
        obs_spaces = self.env.observation_space.spaces

        obs_dim = sum(space.shape[0] for space in obs_spaces.values())

        self.goal_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(2,),
            dtype=np.float32
        )

        self.observation_space = spaces.Dict({
            'observation': spaces.Box(
                low=-np.inf,
                high=np.inf,
                shape=(obs_dim,),
                dtype=np.float32
            ),
            'achieved_goal': self.goal_space,
            'desired_goal': self.goal_space,
        })

        self.action_space = self.env.action_space

    def _flatten_obs(self, obs_dict):
        return np.concatenate([np.asarray(obs_dict[k], dtype=np.float32) for k in self.obs_keys], axis=0)

    def _extract_achieved_goal(self, obs_dict):
        return np.asarray(obs_dict['P'], dtype=np.float32)[:2]

    def _desired_goal(self):
        return np.asarray(self.env._goal_xy, dtype=np.float32)

    def reset(self, seed=None, options=None):
        obs_dict, info = self.env.reset(history=False, seed=seed)

        flat_obs = self._flatten_obs(obs_dict)
        achieved = self._extract_achieved_goal(obs_dict)
        desired = self._desired_goal()

        goal_obs = {
            'observation': flat_obs,
            'achieved_goal': achieved,
            'desired_goal': desired,
        }
        return goal_obs, info

    def step(self, action):
        obs_dict, reward, terminated, truncated, info = self.env.step(
            action,
            history=False,
            show_reward=True
        )

        flat_obs = self._flatten_obs(obs_dict)
        achieved = self._extract_achieved_goal(obs_dict)
        desired = self._desired_goal()

        reward = self.compute_reward(achieved, desired, info)

        goal_obs = {
            'observation': flat_obs,
            'achieved_goal': achieved,
            'desired_goal': desired,
        }
        return goal_obs, reward, terminated, truncated, info

    def compute_reward(self, achieved_goal, desired_goal, info):
        ag = np.asarray(achieved_goal, dtype=np.float32)
        dg = np.asarray(desired_goal, dtype=np.float32)

        diff = ag - dg
        if diff.ndim == 1:
            dist = np.linalg.norm(diff)
        else:
            dist = np.linalg.norm(diff, axis=-1)

        success = (dist <= self.success_radius).astype(np.float32)
        return -1.0 + success

In [3]:
from stable_baselines3.common.callbacks import BaseCallback

class EpisodeRewardCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_count = 0

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        for info in infos:
            if "episode" in info:
                self.episode_count += 1
                ep_reward = info["episode"]["r"]
                ep_length = info["episode"]["l"]
                self.episode_rewards.append(ep_reward)
                if self.verbose > 0 and (self.episode_count % 100 == 0):
                    print(f"Episode {self.episode_count}: reward: {ep_reward}, length: {ep_length}")
        return True

reward_callback = EpisodeRewardCallback(verbose=1)

In [ ]:
LOGDIR = '/home/et2842/causal/antmaze_expert'
os.makedirs(LOGDIR, exist_ok=True)

NUM_ENVS = 32

def make_env(rank):
    def _init():
        env = AntMazePCH().env
        env = AntMazeSCMGoalWrapper(env)
        env = Monitor(env)
        return env
    return _init``

vec_env = SubprocVecEnv([make_env(i) for i in range(NUM_ENVS)])

model = SAC(
    policy='MultiInputPolicy',
    env=vec_env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy=GoalSelectionStrategy.FUTURE,
    ),
    device='cuda',
    learning_starts=1000*NUM_ENVS+1,
    verbose=0,
    batch_size=512,
    gamma=0.99,
    train_freq=1,
    gradient_steps=1,
    learning_rate=3e-4,
    tau=0.005,
    tensorboard_log=os.path.join(LOGDIR, 'tb'),
)

model.learn(total_timesteps=int(10_000_000), progress_bar=True, callback=reward_callback)
model.save(os.path.join(LOGDIR, 'sac_her_antmaze_scm_expert'))

vec_env.close()
print('Training finished and model saved.')

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_D

Output()

Episode 100: reward: -1000.0, length: 1000

Episode 200: reward: -1000.0, length: 1000

Episode 300: reward: -1000.0, length: 1000

...

Episode 1800: reward: -1000.0, length: 1000

Episode 1900: reward: -1000.0, length: 1000

Episode 2000: reward: -1000.0, length: 1000

Episode 2100: reward: -1000.0, length: 1000

Episode 2200: reward: -1000.0, length: 1000

In [ ]:
import numpy as np
import os
import gymnasium as gym
import matplotlib.pyplot as plt

from stable_baselines3 import SAC
from stable_baselines3.common.utils import set_random_seed
from causal_gym import AntMazePCH

MODEL_PATH = '/home/et2842/causal/antmaze_expert/sac_her_antmaze_scm_expert.zip'
assert os.path.exists(MODEL_PATH), f'Model file not found: {MODEL_PATH}'

def make_eval_env(seed=0):
    scm = AntMazePCH(seed=seed).env
    env = AntMazeSCMGoalWrapper(scm)
    return env

env = make_eval_env(seed=123)
model = SAC.load(MODEL_PATH, env=env, device='cuda')

In [ ]:
obs, info = env.reset()
obs, info

In [ ]:
action, _ = model.predict(obs, deterministic=True)
obs, reward, done, truncated, info = env.step(action)
action, obs, reward, done, truncated, info